# Lab 4.3 &mdash; Multi-Tool Orchestration

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Decide which tools an unattended agent may be handed at all
- Answer a <code>tool_call</code> with the <code>ToolMessage</code> it is waiting for
- Recognise a repeated call, because that is what a stuck agent looks like from outside
- Write the tool-calling loop yourself, then watch the model sequence two tools

> **How this lab works.** You write real LangChain and MCP code. Fill every `BLANK`, then run
> the **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a `@tool`, an argument schema, a `ToolMessage`, an `mcp.types.Tool`), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code
> in front of the sandbox model; that is the part worth watching. The score line is feedback,
> not a grade.

> **Builds on Lab 4.1's contract.** A tool that returns instead of raising is what makes
> a multi-step run recoverable; here you find out what still goes wrong when it does.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and that reasoning is billed as completion
# tokens. It is off here because tool selection is a short decision and you will make a lot
# of them today. Pass think=True to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 4 labs -- the same payment exceptions as Day 1,
# now reached through tools the model chooses, and then through tools you did not write.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the toolkit (nothing to fill in)
# Four tools over that ledger, written with LangChain's @tool decorator. Three read; one
# moves money -- the distinction that starts mattering the moment a model is choosing.
# Read the docstrings properly: they are not comments, they are the API the model sees.
from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you already have the reference. Not for searching across payments --
    use search_payments when you do not have one.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


@tool
def search_payments(counterparty: str = "", status: str = "") -> str:
    """Return every ledger record matching a counterparty, a status, or both.

    Use when you must find which payments match. Not for one known reference --
    use lookup_payment for that.
    """
    hits = [{"ref": r, **v} for r, v in LEDGER.items()
            if (not counterparty or v["counterparty"] == counterparty)
            and (not status or v["status"] == status)]
    return json.dumps(hits)


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code such as 'LIMIT_BREACH'.

    Use once you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


@tool
def release_payment(ref: str) -> str:
    """Release one held payment so that it settles. This one moves money.

    Use only after a named human has approved this specific release. Not for reading,
    searching or explaining.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, "released": True, "was": record["status"]})


TOOLKIT = [lookup_payment, search_payments, policy_for, release_payment]
BY_NAME = {t.name: t for t in TOOLKIT}
print("toolkit:", ", ".join(BY_NAME))

## Concept

&ldquo;Multi-tool&rdquo; is three separate problems wearing one name:

1. **Selection** &mdash; which tool. Lab 4.2 measured this.
2. **Arguments** &mdash; what to pass. The model extracts them from a request a human wrote.
3. **Sequencing** &mdash; the interesting one. `policy_for` needs a reason code that only
   `lookup_payment` can supply, so step two's argument does not exist until step one has run.

Nothing coordinates those three but the message list. You send messages, the model replies with
`tool_calls`, you run them and append a `ToolMessage` for each, and you send the lot back. That
loop is the whole of `create_agent`, minus the hardening.

Plus the failure that ends production agents: a loop that does not end. Two calls with the same
tool and the same arguments cannot produce different answers, so the second one is always wasted
&mdash; and the tenth one is an incident.

## Section 1 &mdash; Which tools does it get?

Before any loop runs, someone decides what is in reach. Four tools exist; this agent investigates
and reports, and nothing in this lab approves anything.

A tool the model cannot see is a tool it cannot call. It is the cheapest control in the module,
and it is a decision, not a default.

In [ ]:
def tools_for_investigation() -> list:
    """The tools an unattended investigation agent may be handed.

    All four in BY_NAME exist and all four work. That is not the question.
    """
    # TODO: return the list of tool objects this agent should get.
    #       One of the four moves money, and nothing here approves anything.
    return BLANK

In [ ]:
# --- Self-check: Section 1   (tool objects only -- no model call)
def _names():
    return {t.name for t in tools_for_investigation()}

check("the agent can read a payment and read the policy",
      lambda: {"lookup_payment", "policy_for"} <= _names(),
      "without both of these it cannot answer a single question in this lab")
check("it is NOT handed the tool that moves money",
      lambda: "release_payment" not in _names(),
      "a tool the model cannot see is a tool it cannot call -- the cheapest control there is")
check("every entry is a real tool object, not a bare function",
      lambda: all(hasattr(t, "name") and hasattr(t, "invoke")
                  for t in tools_for_investigation()))
check("they are the toolkit's own objects, not copies",
      lambda: all(t is BY_NAME[t.name] for t in tools_for_investigation()))

guard(lambda: print("  handed to the agent:", ", ".join(sorted(_names()))))

## Section 2 &mdash; Answering a tool call

The model asks for a tool by emitting a `tool_call`: a dict with a `name`, an `args` and an `id`.
You run it and reply with a `ToolMessage`.

The `id` is the part people get wrong. A single turn can carry several calls at once, and the
`ToolMessage` says which one it is answering. Get it wrong and the model reads the policy text as
the answer to the payment lookup, with nothing raised anywhere.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

def run_one_call(call: dict, tools: list) -> ToolMessage:
    """Run one tool call the model asked for, and package the result as its reply.

    `call` is one entry of AIMessage.tool_calls: {"name", "args", "id", "type"}.
    """
    by_name = {t.name: t for t in tools}
    result = by_name[call["name"]].invoke(call["args"])
    return ToolMessage(
        content=str(result),
        # TODO: a turn can have several calls in flight at once. What pairs THIS result
        #       to the request that asked for it?
        tool_call_id=BLANK,
    )

In [ ]:
# --- Self-check: Section 2   (real messages and real tools -- no model call)
def _call(name="lookup_payment", args=None, cid="call_1") -> dict:
    """One tool_call, shaped exactly as the model emits it."""
    return {"name": name, "args": args if args is not None else {"ref": "PMT-1002"},
            "id": cid, "type": "tool_call"}

check("the tool actually ran, and its output came back as text",
      lambda: "INSUFFICIENT_FUNDS" in run_one_call(_call(), TOOLKIT).content)
check("the reply is a ToolMessage -- the only message type that answers a call",
      lambda: isinstance(run_one_call(_call(), TOOLKIT), ToolMessage))
check("the reply is addressed to the call that asked for it",
      lambda: run_one_call(_call(cid="call_7"), TOOLKIT).tool_call_id == "call_7",
      "two calls in flight and a wrong id here silently answers the wrong question")
check("two calls get two different addresses",
      lambda: run_one_call(_call(cid="a"), TOOLKIT).tool_call_id
              != run_one_call(_call(cid="b"), TOOLKIT).tool_call_id)
check("a different tool is answered the same way",
      lambda: "Treasury approval" in
              run_one_call(_call("policy_for", {"reason_code": "LIMIT_BREACH"}), TOOLKIT).content)
check("the content is a string, whatever the tool returned",
      lambda: isinstance(run_one_call(_call("search_payments",
                                            {"counterparty": "NORTHWIND"}), TOOLKIT).content, str))

## Section 3 &mdash; Knowing you have been here before

Two calls that are the same in every way return the same answer. So the second one buys nothing,
and an agent that keeps making it is stuck rather than slow.

The trap is the `id`: it is new on every single call, so an identity that includes it never
matches anything and the check silently never fires.

In [ ]:
def call_key(call: dict):
    """An identity for one call, so that a repeat of it is recognisable.

    json.dumps(..., sort_keys=True) makes two argument dicts compare equal whatever order
    the model happened to write the keys in.
    """
    # TODO: what makes two calls THE SAME call? Look hard at what is in a tool_call
    #       that changes every single turn.
    return BLANK

In [ ]:
# --- Self-check: Section 3   (pure identity -- no model call)
check("two identical calls are the same call",
      lambda: call_key(_call()) == call_key(_call()))
check("the id is NOT part of what makes a call the same",
      lambda: call_key(_call(cid="a")) == call_key(_call(cid="b")),
      "the id is new every turn -- include it and no repeat is ever detected")
check("argument order does not make a call look new",
      lambda: call_key(_call("search_payments", {"counterparty": "ZENITH", "status": "held"}))
              == call_key(_call("search_payments", {"status": "held", "counterparty": "ZENITH"})))
check("a different argument is a different call",
      lambda: call_key(_call(args={"ref": "PMT-1002"}))
              != call_key(_call(args={"ref": "PMT-1003"})))
check("a different tool is a different call",
      lambda: call_key(_call("lookup_payment"))
              != call_key(_call("policy_for", {"reason_code": "LIMIT_BREACH"})))
check("the key is hashable, because it goes in a set",
      lambda: len({call_key(_call()), call_key(_call(cid="z"))}) == 1)

## Section 4 &mdash; The loop

Nothing left to fill in. Read it once: this is `create_agent` with the hardening taken out, and
every line of it is one of the three problems from the concept.

Two exits besides finishing: a repeated call, and a budget. The budget catches the runs a repeat
check misses &mdash; no repeat, just a plan that will not end.

In [ ]:
INVESTIGATE_SYSTEM = (
    "You are a payments operations analyst. Use the tools to find out what happened and what "
    "the policy says about it. When you have the answer, reply in one sentence without calling "
    "a tool.")

def investigate(request: str, max_calls: int = 4) -> dict:
    """The tool-calling loop, written out. Always returns an outcome and a trace."""
    tools = tools_for_investigation()
    bound = get_llm().bind_tools(tools)
    messages = [SystemMessage(INVESTIGATE_SYSTEM), HumanMessage(request)]
    seen, trace = set(), []

    while True:
        reply = bound.invoke(messages)
        messages.append(reply)

        if not reply.tool_calls:                       # it answered instead of asking
            return {"outcome": "answered", "answer": reply.content, "trace": trace}

        for call in reply.tool_calls:
            if call_key(call) in seen:                 # it has been here before
                return {"outcome": "loop", "trace": trace,
                        "answer": f"{call['name']} was already called with these arguments"}
            seen.add(call_key(call))
            messages.append(run_one_call(call, tools))
            trace.append((call["name"], call["args"]))

        if len(trace) >= max_calls:                    # it is not converging
            return {"outcome": "budget", "trace": trace,
                    "answer": f"stopped after {len(trace)} calls without an answer"}

## Run it for real

Three requests. The first needs two tools in sequence; the second needs one; the third names no
payment at all, and there is no honest answer to it.

In [ ]:
if llm_ready():
    def _run():
        for request in ("Why did PMT-1002 fail, and what should we do about it?",
                        "Which payments are currently held for NORTHWIND?",
                        "Why did it fail?"):
            out = investigate(request)
            print(f"\n  {request}")
            for name, args in out["trace"]:
                print(f"    -> {name}({args})")
            print(f"    [{out['outcome']}] {str(out['answer'])[:170]}")
    guard(_run)

### Read it

**The first request is the whole point of the lab.** Watch the trace: `lookup_payment` first, then
`policy_for` with `reason_code=INSUFFICIENT_FUNDS` &mdash; an argument that did not exist when the
run started. Nothing in your code threaded it through. The model read the first tool's result out
of the message list and used it to write the second call. That is sequencing, and it is the reason
the message list is the whole architecture.

**The third request has no answer, and watch what happens anyway.** The honest reply is a
question: *which payment?* Depending on the turn you will see it invent a reference, call
`search_payments` with nothing, or ask. Only the last is right, and nothing in this design asks
for it.

That is failure 4 from the deck &mdash; the one that looks like success. Every step returned
cleanly, the loop terminated, the outcome says `answered`, and the answer is worthless. No loop
check catches it and no budget catches it. Module 5 gives it a home: a supervisor whose job
includes deciding that *neither* plan applies.

In [ ]:
score()

## Your turn

1. Set `max_calls=1` and re-run the first request. The outcome changes to `budget` with a partial
   trace. What should the agent tell the user &mdash; and how is that different from an error?
2. Add `release_payment` to `tools_for_investigation()` and ask &ldquo;Treasury approved PMT-1003,
   release it.&rdquo; Then take it out again. That two-line diff is the difference between an agent
   that reports and an agent that acts.
3. The repeat check compares exact arguments, so an agent that alternates `PMT-1002`, `PMT-1003`,
   `PMT-1002` defeats it. Widen it to a repeat *within a window* and see what it costs you in
   false positives on a legitimate multi-payment investigation.
4. Replace the whole loop with `create_agent(model=get_llm(), tools=tools_for_investigation(),
   system_prompt=INVESTIGATE_SYSTEM)` and compare the traces. What did you give up, and what did
   you get?